In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_66_try_0.pickle

In [ ]:
%%RecordEvent
%%time
### cell 67 ###

def grab_subset_of_data_79(df, question_of_interest):
    # Select columns on the GPU, extract the suffix after '-', strip whitespace, and drop all-null rows
    subset = df.filter(like=question_of_interest)
    subset.columns = (
        subset.columns
              .str.split('-')
              .str.get(-1)
              .str.strip()
    )
    return subset.dropna(how='all')

# Bundle the 4 years of DataFrames into a dict for iteration
year_dfs = {
    2019: responses_df_2019_cell10,
    2020: responses_df_2020,
    2021: responses_df_2021,
    2022: responses_df_2022_cell10,
}

tpus_percentages = {}
for year, df in year_dfs.items():
    # Harmonize the question text in column names (GPU string replace)
    df = df.rename(
        columns=lambda col: col.replace(
            'Which types of specialized hardware do you use on a regular basis?',
            question_of_interest_cell79
        )
    )
    hw_df = grab_subset_of_data_79(df, question_of_interest_cell79)
    # Count non-null, compute percentage
    pct = hw_df.count() * 100 / len(hw_df)
    # Unify any “Google Cloud TPUs” labels into “TPUs” and strip whitespace
    pct.index = pct.index.str.replace('Google Cloud TPUs', 'TPUs').str.strip()
    tpus_percentages[year] = pct['TPUs']

# Inspect each year's TPU percentage series
for year, pct in tpus_percentages.items():
    print(f"{year} TPU percentage:")
    pct.info()

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_67_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_67_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[67], f)


In [ ]:
opt_output = Out.get(4)